In [37]:
import torch
import numpy as np
from transformers import BertTokenizer
import json
import pandas as pd
from torch import nn
from transformers import BertModel
from torch.optim import Adam
from tqdm import tqdm
from ipywidgets import FloatProgress
from torch.utils.tensorboard import SummaryWriter
import re
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn import metrics
import json
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Please download the BERT weight files from **[https://huggingface.co/google-bert/bert-base-chinese](https://huggingface.co/google-bert/bert-base-chinese)** and store them in the `model` directory.

In [38]:
BERT_PATH = './model/bert-base-chinese'
tokenizer = BertTokenizer.from_pretrained(BERT_PATH)

In [39]:
def pad_to_512(input_string, max_pad_lenth=512):
    while len(input_string) < max_pad_lenth:
        input_string.append(int(-100))
    return input_string

In [40]:
intents_num = {'直线距离查询': 0,
            '步行距离查询': 1,
            '车行距离查询': 2,
            '步行时间查询': 3,
            '开车时间查询': 4,
            '骑车时间查询': 5,
            '公共交通时间查询': 6,
            'POI查询': 7,
            '小区查询': 8,
            '小区均价查询': 9,
            '小区绿化率查询': 10,
            '高峰期开车时间查询': 11,
            '高峰期公共交通时间查询': 12,
            '非高峰期开车时间查询': 13,
            '非高峰期公共交通时间查询': 14,
            '通勤时间查询':15
          }

slots_num = {'O': 0,
          'B-city': 1,
          'I-city': 2,
          'B-district': 3,
          'I-district': 4,
          'B-community': 5,
          'I-community': 6,
          'B-POI': 7,
          'I-POI': 8,
          'B-POI:shopping_mall': 9,
          'I-POI:shopping_mall': 10,
          'B-POI:supermarket': 11,
          'I-POI:supermarket': 12,
          'B-POI:hospital': 13,
          'I-POI:hospital': 14,
          'B-POI:park': 15,
          'I-POI:park': 16,
          'B-POI:subway_station': 17,
          'I-POI:subway_station': 18,
          'B-POI:bus_stop': 19,
          'I-POI:bus_stop': 20,
          'B-POI:primary_school': 21,
          'I-POI:primary_school': 22,
          'B-POI:middle_school': 23,
          'I-POI:middle_school': 24,
          'B-POI_third': 25,
          'I-POI_third': 26,
          'B-sale_status': 27,
          'I-sale_status': 28,
          'B-community_tag': 29,
          'I-community_tag': 30,
          'B-distance': 31,
          'I-distance': 32,
          'B-house_price': 33,
          'I-house_price': 34,
          'B-green_rate': 35,
          'I-green_rate': 36,
          }

### Read the training, testing, and validation sets.

In [41]:
def json2dataframe(all_datas):
    df = pd.DataFrame(columns=['category', 'text', 'intents', 'slots'])
    intents = []
    for data in all_datas:
        query = data['query']
        slots = data['slots'].split(' ')
        encoded_text = tokenizer(query, return_tensors='pt')
        tokens = tokenizer.convert_ids_to_tokens(encoded_text['input_ids'][0])
        slots = data['tokens_slots(bert)']
        numbered_slots = [slots_num[item] for item in slots]
        intents_label = [0.0] * len(intents_num)
        if data['intent'] not in intents and '+' not in data['intent']:
            intents.append(data['intent'])
        if '+' in data['intent']:
            data['intent'] = data['intent'].split('+')
            for intent in data['intent']:
                intents_label[intents_num[intent]] = 1.0
            df = pd.concat([df, pd.DataFrame([{'category': intents_label, 'text': data['query'], 'intents': 1, 'slots': numbered_slots}])], ignore_index=True)
        elif '+' not in data['intent']:
            intent = data['intent']
            intents_label[intents_num[intent]] = 1.0
            df = pd.concat([df, pd.DataFrame([{'category': intents_label, 'text': data['query'], 'intents': 0, 'slots': numbered_slots}])], ignore_index=True)
    df['slots'] = df['slots'].apply(pad_to_512)
    return df

In [42]:
train_path = './dataset/train.json'
with open(train_path, 'r', encoding='utf-8') as f:
    all_datas = json.load(f)
df_train = json2dataframe(all_datas)

validation_path = './dataset/val.json'
with open(validation_path, 'r', encoding='utf-8') as f:
    all_datas = json.load(f)
df_val = json2dataframe(all_datas)

test_path = './dataset/test.json'
with open(test_path, 'r', encoding='utf-8') as f:
    all_datas = json.load(f)
df_test = json2dataframe(all_datas)

print(len(df_train), len(df_val), len(df_test))

23407 2920 2943


In [43]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.labels = df['category']
        self.texts = [tokenizer(text, 
                                padding='max_length', 
                                max_length = 512, 
                                truncation=True,
                                return_tensors="pt") 
                      for text in df['text']]
        self.num_intents = df['intents']
        self.slots = df['slots']

    def classes(self):
        return self.labels

    def __len__(self):
        return len(self.labels)

    def get_batch_labels(self, idx):
        # Fetch a batch of labels
        return np.array(self.labels[idx])

    def get_batch_texts(self, idx):
        # Fetch a batch of inputs
        return self.texts[idx]

    def get_batch_num_intents(self, idx):
        # Fetch a batch of inputs
        return np.array(self.num_intents[idx])

    def get_batch_slots(self, idx):
        # Fetch a batch of inputs
        return np.array(self.slots[idx])

    def __getitem__(self, idx):
        batch_texts = self.get_batch_texts(idx)
        batch_y = self.get_batch_labels(idx)
        batch_num = self.get_batch_num_intents(idx)
        batch_slots = self.get_batch_slots(idx)
        return batch_texts, batch_y, batch_num, batch_slots

### Build the neural network.

In [44]:
class BertClassifier(nn.Module):
    def __init__(self, dropout=0.5):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-chinese')
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(768, 16)
        self.linear2 = nn.Linear(768, 2)
        self.linear3 = nn.Linear(768, 37)
        self.sigmoid = nn.Sigmoid()
        self.softmax = nn.Softmax()

    def forward(self, input_id, mask):
        last_hidden_state, pooled_output = self.bert(input_ids= input_id, attention_mask=mask,return_dict=False)
        dropout_output = self.dropout(pooled_output)
        linear1_output = self.linear1(dropout_output)
        intent_probability = self.sigmoid(linear1_output)
        num_intents = self.linear2(dropout_output)
        last_hidden_state_output = self.dropout(last_hidden_state)
        slot_probability = self.linear3(last_hidden_state_output)
        return intent_probability, num_intents, slot_probability

### Train

In [45]:
def tensors_equal_ignore_order(tensor1, tensor2):
    # Sort the two tensors along the specified dimension
    sorted_tensor1, _ = torch.sort(tensor1)
    sorted_tensor2, _ = torch.sort(tensor2)
    results = []
    for row1, row2 in zip(sorted_tensor1, sorted_tensor2):
        results.append(torch.equal(row1, row2))
        results_tensor = torch.tensor(results, dtype=torch.bool)
    return results_tensor

# Input two tensors: the first tensor is the probability tensor, and the second tensor is the one-hot encoded label tensor.
def compute_multi_label_acc(probility, label):
    probility, idx1 = torch.sort(probility, descending=True)
    label, idx2 = torch.sort(label, descending=True)
    idx1 = idx1[:,0:2]
    idx2 = idx2[:,0:2]
    for i,labl in enumerate(label):
        if labl.sum() < 2:
            idx1[i,1] = 0
            idx2[i,1] = 0
    acc = tensors_equal_ignore_order(idx1, idx2).sum().item()
    return acc

In [46]:
def train(model, train_data, val_data, learning_rate, epochs):
    # Sort the two tensors along the specified dimension using the Dataset class to retrieve the training and validation sets
    train, val = Dataset(train_data), Dataset(val_data)
    # Use DataLoader to retrieve data based on batch_size, and shuffle samples during training
    train_dataloader = torch.utils.data.DataLoader(train, batch_size=5, shuffle=True)
    val_dataloader = torch.utils.data.DataLoader(val, batch_size=5)
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    # loss
    criterion = nn.CrossEntropyLoss()
    binary_criterion = nn.BCELoss()
    optimizer = Adam(model.parameters(), lr=learning_rate)
    if use_cuda:
        model = model.cuda()
        criterion = criterion.cuda()
    for epoch_num in range(epochs):
        total_intent_acc_train = 0
        total_num_acc_train = 0
        total_loss_train = 0
        total_slot_acc_train = 0
        total_tokens_train = 0
        for train_input, train_intent_label, train_num_label, train_slot_label in tqdm(train_dataloader):
            train_intent_label = train_intent_label.to(device)
            train_num_label = train_num_label.to(device)
            train_slot_label = train_slot_label.to(device)
            mask = train_input['attention_mask'].squeeze(1).to(device)
            input_id = train_input['input_ids'].squeeze(1).to(device)
            # model output
            intent_probability, num_intents, slot_probability = model(input_id, mask)
            # compute loss
            intent_loss = binary_criterion(intent_probability, train_intent_label.float())
            active_loss = mask.view(-1) == 1
            active_logits = slot_probability.view(-1, 37)[active_loss]
            active_labels = train_slot_label.view(-1)[active_loss]
            slot_loss = criterion(active_logits, active_labels.long())
            num_loss = criterion(num_intents, train_num_label.long())
            loss = intent_loss + num_loss + slot_loss
            total_loss_train += loss.item()
            # compute metric
            intent_acc = compute_multi_label_acc(intent_probability, train_intent_label)
            total_intent_acc_train += intent_acc
            num_intent_acc = (num_intents.argmax(dim=1) == train_num_label).sum().item()
            total_num_acc_train += num_intent_acc
            word_leval_slots_acc = (slot_probability.argmax(dim=2).view(-1)[active_loss] == train_slot_label.view(-1)[active_loss]).sum().item()
            batch_token_nums = active_loss.sum().item()
            total_slot_acc_train += word_leval_slots_acc
            total_tokens_train += batch_token_nums
            model.zero_grad()
            loss.backward()
            optimizer.step()
        # ------- val -----------
        total_intent_acc_val = 0
        total_num_acc_val = 0
        total_loss_val = 0
        total_slot_acc_val = 0
        total_tokens_val = 0
        with torch.no_grad():
            for val_input, val_intent_label, val_num_label, val_slot_label in val_dataloader:
                val_intent_label = val_intent_label.to(device)
                val_num_label = val_num_label.to(device)
                val_slot_label = val_slot_label.to(device)
                mask = val_input['attention_mask'].squeeze(1).to(device)
                input_id = val_input['input_ids'].squeeze(1).to(device)
                intent_probability, num_intents, slot_probability = model(input_id, mask)
                intent_loss = binary_criterion(intent_probability, val_intent_label.float())
                # compute slots loss
                active_loss = mask.view(-1) == 1
                active_logits = slot_probability.view(-1, 37)[active_loss]
                active_labels = val_slot_label.view(-1)[active_loss]
                slot_loss = criterion(active_logits, active_labels.long())
                # compute intent num loss
                num_loss = criterion(num_intents, val_num_label.long())
                loss = intent_loss + num_loss + slot_loss
                total_loss_val += loss.item()
                # compute metric
                intent_acc = compute_multi_label_acc(intent_probability, val_intent_label)
                total_intent_acc_val += intent_acc
                num_intent_acc = (num_intents.argmax(dim=1) == val_num_label).sum().item()
                total_num_acc_val += num_intent_acc
                word_leval_slots_acc = (slot_probability.argmax(dim=2).view(-1)[active_loss] == val_slot_label.view(-1)[active_loss]).sum().item()
                batch_token_nums = active_loss.sum().item()
                total_slot_acc_val += word_leval_slots_acc
                total_tokens_val += batch_token_nums
        writer.add_scalar('Loss/train', total_loss_train / len(train_data), epoch_num)
        writer.add_scalar('Accuracy/train_intent', total_intent_acc_train / len(train_data), epoch_num)
        writer.add_scalar('Accuracy/train_num_intents', total_num_acc_train / len(train_data), epoch_num)
        writer.add_scalar('Accuracy/train_token_level_slot_acc', total_slot_acc_train / total_tokens_train, epoch_num)
        writer.add_scalar('Loss/val', total_loss_val / len(val_data), epoch_num)
        writer.add_scalar('Accuracy/val_intent', total_intent_acc_val / len(val_data), epoch_num)
        writer.add_scalar('Accuracy/val_num_intents', total_num_acc_val / len(val_data), epoch_num)
        writer.add_scalar('Accuracy/val_token_level_slot_acc', total_slot_acc_val / total_tokens_val, epoch_num)

        print(
            f'''Epochs: {epoch_num + 1} 
            | Train Loss: {total_loss_train / len(train_data): .3f} 
            | Train Intent Accuracy: {total_intent_acc_train / len(train_data): .3f}
            | Train Num of intents Accuracy: {total_num_acc_train / len(train_data): .3f} 
            | Train Token-level Slots Accuracy: {total_slot_acc_train / total_tokens_train: .3f} 
            | Val Loss: {total_loss_val / len(val_data): .3f} 
            | Val Intent Accuracy: {total_intent_acc_val / len(val_data): .3f}
            | Val Num of intents Accuracy: {total_num_acc_val / len(val_data): .3f}
            | Val Token-level Slots Accuracy: {total_slot_acc_val / total_tokens_val: .3f} ''')
        writer.close()

### Begin train

In [47]:
EPOCHS = 10
writer = SummaryWriter('./runs')
model = BertClassifier()
LR = 1e-6

In [48]:
train(model, df_train, df_val, LR, EPOCHS)

100%|██████████| 4682/4682 [05:59<00:00, 13.04it/s]


Epochs: 1 
            | Train Loss:  0.128 
            | Train Intent Accuracy:  0.611
            | Train Num of intents Accuracy:  0.994 
            | Train Token-level Slots Accuracy:  0.921 
            | Val Loss:  0.046 
            | Val Intent Accuracy:  0.891
            | Val Num of intents Accuracy:  1.000
            | Val Token-level Slots Accuracy:  0.977 


100%|██████████| 4682/4682 [06:01<00:00, 12.96it/s]


Epochs: 2 
            | Train Loss:  0.032 
            | Train Intent Accuracy:  0.957
            | Train Num of intents Accuracy:  1.000 
            | Train Token-level Slots Accuracy:  0.982 
            | Val Loss:  0.022 
            | Val Intent Accuracy:  0.984
            | Val Num of intents Accuracy:  1.000
            | Val Token-level Slots Accuracy:  0.985 


100%|██████████| 4682/4682 [06:02<00:00, 12.93it/s]


...

Epochs: 8 
            | Train Loss:  0.003 
            | Train Intent Accuracy:  0.999
            | Train Num of intents Accuracy:  1.000 
            | Train Token-level Slots Accuracy:  0.996 
            | Val Loss:  0.005 
            | Val Intent Accuracy:  1.000
            | Val Num of intents Accuracy:  1.000
            | Val Token-level Slots Accuracy:  0.995 


100%|██████████| 4682/4682 [06:01<00:00, 12.94it/s]


Epochs: 9 
            | Train Loss:  0.003 
            | Train Intent Accuracy:  1.000
            | Train Num of intents Accuracy:  1.000 
            | Train Token-level Slots Accuracy:  0.997 
            | Val Loss:  0.004 
            | Val Intent Accuracy:  1.000
            | Val Num of intents Accuracy:  1.000
            | Val Token-level Slots Accuracy:  0.995 


100%|██████████| 4682/4682 [06:03<00:00, 12.90it/s]


Epochs: 10 
            | Train Loss:  0.002 
            | Train Intent Accuracy:  1.000
            | Train Num of intents Accuracy:  1.000 
            | Train Token-level Slots Accuracy:  0.997 
            | Val Loss:  0.004 
            | Val Intent Accuracy:  1.000
            | Val Num of intents Accuracy:  1.000
            | Val Token-level Slots Accuracy:  0.995 


### Test

In [49]:
def evaluate(model, test_data):
    test = Dataset(test_data)
    test_dataloader = torch.utils.data.DataLoader(test, batch_size=4)
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        model = model.cuda()
    total_intent_acc_test = 0
    total_num_acc_test = 0
    total_tokens_test = 0
    total_slot_acc_test = 0
    with torch.no_grad():
        for test_input, test_intent_label, test_num_label, test_slot_label in test_dataloader:
            test_intent_label = test_intent_label.to(device)
            test_num_label = test_num_label.to(device)
            test_slot_label =  test_slot_label.to(device)
            mask = test_input['attention_mask'].squeeze(1).to(device)
            input_id = test_input['input_ids'].squeeze(1).to(device)
            intent_probability, num_intents, slot_probability = model(input_id, mask)
            active_loss = mask.view(-1) == 1
            intent_acc = compute_multi_label_acc(intent_probability, test_intent_label)
            # intent_acc = (intent_probability.argmax(dim=1) == test_intent_label.argmax(dim=1)).sum().item()
            total_intent_acc_test += intent_acc
            num_intent_acc = (num_intents.argmax(dim=1) == test_num_label).sum().item()
            total_num_acc_test += num_intent_acc
            word_leval_slots_acc = (slot_probability.argmax(dim=2).view(-1)[active_loss] == test_slot_label.view(-1)[active_loss]).sum().item()
            batch_token_nums = active_loss.sum().item()
            total_slot_acc_test += word_leval_slots_acc
            total_tokens_test += batch_token_nums
    print(f'Test Intent Accuracy: {total_intent_acc_test*100 / len(test_data): .2f}%')
    print(f'Test Num of Intent Accuracy: {total_num_acc_test*100 / len(test_data): .2f}%')
    print(f'Test Token-level Slots Accuracy: {total_slot_acc_test*100 / total_tokens_test: .2f}%')

In [50]:
evaluate(model, df_test)

Test Intent Accuracy:  99.97%
Test Num of Intent Accuracy:  100.00%
Test Token-level Slots Accuracy:  99.43%


In [51]:
model_path = './model/bert.pt'
torch.save(model, model_path)

# Predict

In [52]:
def top2_indices(tensor):
    if len(tensor) < 2:
        raise ValueError("Input tensor must have at least 2 elements")
    _, indices = torch.topk(tensor, k=2, dim=0)
    return indices

def find_key(dictionary, value):
    return [key for key, val in dictionary.items() if val == value]

In [53]:
model_path = './model/bert.pt'
model = torch.load(model_path)
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
if use_cuda:
    model = model.cuda()
BERT_PATH = './model/bert-base-chinese'
tokenizer = BertTokenizer.from_pretrained(BERT_PATH)

### Transform slots into keywords

In [54]:
def align_tokens_with_query(tokens, query, query_slot):
    query = list(query)
    new_tokens = []
    origin_slot = query_slot
    if isinstance(origin_slot[0], type('str')):
        origin_slot = origin_slot[0].split(' ')
    new_slot = []
    
    for i, token in enumerate(tokens):
        #print(f'token={token},query[0]={query[0]}')
        #print(f'token={token}')
        #print(f'origin_slot={origin_slot}')
        if token == '[CLS]' or token == '[SEP]':
            continue
        elif token == query[0]:
            new_tokens.append(token)
            new_slot.append(origin_slot[0])
            query = query[1:]
            origin_slot = origin_slot[1:]
        elif '##' in token:
            token = token[2:]
            new_tokens.append(token)
            new_slot.append(origin_slot[0])
            for t in list(token):
                if t == query[0]:
                    query = query[1:]
                    origin_slot = origin_slot[1:]
                    
        elif '[UNK]' == token:
            #print(f'tokens={tokens}')
            for j in range(i+1, len(tokens)):
                if len(tokens[j]) == 1:
                    break
            end_index = query.index(tokens[j])
            unk = ''.join(query[0:end_index])
            new_tokens.append(unk)
            new_slot.append(origin_slot[0])
            query = query[end_index:]
            origin_slot = origin_slot[1:]

        elif len(token)>1:
            new_tokens.append(token)
            new_slot.append(origin_slot[0])
            for t in list(token):
                if t == query[0]:
                    query = query[1:]
                    origin_slot = origin_slot[1:]
    return new_tokens, new_slot

def restore_keywords_from_tokens(tokens, token_slot):
    keywords = []
    current_tokens = []
    current_label = None
    token_slot = token_slot[1:-1]

    for token, slot in zip(tokens, token_slot):
        if slot.startswith('B-'):
            if current_tokens:
                keywords.append((''.join(current_tokens), current_label))
                current_tokens = []
            current_label = slot[2:]
            current_tokens.append(token)
        elif slot.startswith('I-') and current_label == slot[2:]:
            current_tokens.append(token)
        else:
            if current_tokens:
                keywords.append((''.join(current_tokens), current_label))
                current_tokens = []
                current_label = None

    if current_tokens:
        keywords.append((''.join(current_tokens), current_label))

    return keywords

# def restore_keywords_from_query(query, slots):
#     keywords = []
#     current_tokens = []
#     current_label = None
#     query = list(query)
#     if slots[0] == '[CLS]':
#         slots = slots[1:-1]

#     for token, slot in zip(query, slots):
#         if slot.startswith('B-'):
#             if current_tokens:
#                 keywords.append((''.join(current_tokens), current_label))
#                 current_tokens = []
#             current_label = slot[2:]
#             current_tokens.append(token)
#         elif slot.startswith('I-') and current_label == slot[2:]:
#             current_tokens.append(token)
#         else:
#             if current_tokens:
#                 keywords.append((''.join(current_tokens), current_label))
#                 current_tokens = []
#                 current_label = None

#     if current_tokens:
#         keywords.append((''.join(current_tokens), current_label))

#     return keywords

In [55]:
def get_char_level_tokens(origin_tokens):
    char_level_tokens = []
    for token in origin_tokens:
        if len(token) == 1:
            char_level_tokens.append(token)
        else:
            for t in list(token):
                char_level_tokens.append(t)
    return char_level_tokens

In [56]:
def extract_entities(sentence, tags):
    if tags[0] == 'O':
        tags = tags[1:-1]
    entities = []
    i = 0
    n = len(tags)

    while i < n:
        if tags[i].startswith("B-"):
            entity_type = tags[i][2:]
            start = i
            i += 1
            while i < n and tags[i] == f"I-{entity_type}":
                i += 1
            word = sentence[start:i]
            # print(word)
            # print(entity_type)
            #entities[word] = entity_type
            entities.append((word, entity_type))
        else:
            i += 1
    return entities


In [57]:
model_path = './model/bert.pt'
model = torch.load(model_path)
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
if use_cuda:
    model = model.cuda()
BERT_PATH = './model/bert-base-chinese'
tokenizer = BertTokenizer.from_pretrained(BERT_PATH)

In [58]:
def intent2label(intents_row):
    intents_num = {'直线距离查询': 0,
            '步行距离查询': 1,
            '车行距离查询': 2,
            '步行时间查询': 3,
            '开车时间查询': 4,
            '骑车时间查询': 5,
            '公共交通时间查询': 6,
            'POI查询': 7,
            '小区查询': 8,
            '小区均价查询': 9,
            '小区绿化率查询': 10,
            '高峰期开车时间查询': 11,
            '高峰期公共交通时间查询': 12,
            '非高峰期开车时间查询': 13,
            '非高峰期公共交通时间查询': 14,
            '通勤时间查询': 15
          }
    intents_label = [0] * len(intents_num)
    if '+' in intents_row:
        intents = intents_row.split('+')
        for intent in intents:
            intents_label[intents_num[intent]] = 1
    elif '+' not in intents_row:
        intent = intents_row
        intents_label[intents_num[intent]] = 1
    return intents_label

def extract_keywords(tokens, token_slot):
    keywords = []
    current_keyword = ''
    current_slot = ''
    
    for token, slot in zip(tokens[1:-1], token_slot[1:-1]):  # Skip [CLS] and [SEP]
        if slot.startswith('B-'):
            if current_keyword:
                keywords.append((current_keyword.strip(), current_slot.split('-')[1]))
            current_keyword = token
            current_slot = slot
        elif slot.startswith('I-') and slot[2:] == current_slot[2:]:
            current_keyword += token
        elif slot == 'O':
            if current_keyword:
                keywords.append((current_keyword.strip(), current_slot.split('-')[1]))
                current_keyword = ''
                current_slot = ''
    
    if current_keyword:
        keywords.append((current_keyword.strip(), current_slot.split('-')[1]))
    
    return keywords

In [59]:
data_file = './dataset/test.json'
with open(data_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)
pred_intent_num = []
true_intent_num = []
pred_intent_label = []
true_intent_label = []
true_key_words = []
pred_key_words = []
true_token_slots = []
pred_token_slots = []
i = 0

for data in tqdm(datas):
    query = data['query']
    # print(query)
    origin_slots = data['slots'].split(',')
    bert_slots = data["tokens_slots(bert)"]
    # print(origin_slots)
    # print(bert_slots)
    true_key_word = extract_entities(query, bert_slots)
    #print(f'true_key_word:{true_key_word}')
    true_key_words.append(true_key_word)
    intent = data['intent']
    if '+' in intent:
        if intent.count('+') == 1:
            true_intent_num.append(1)
    if '+' not in intent:
        true_intent_num.append(0)
    intent_label = intent2label(intent)
    true_intent_label.append(list(intent_label))
    encoded_text = tokenizer(query, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(encoded_text['input_ids'][0])
    # print(query)
    # print(tokens)
    # print(origin_slots)
    new_tokens, _ = align_tokens_with_query(tokens, query, origin_slots)
    char_level_tokens = get_char_level_tokens(new_tokens)
    # print(f"new_slots:{new_slots}")
    # print(f'new_tokens:{new_tokens}')
    # new_slots_to_num = [slots_num[item] for item in new_slots]
    # true_token_slots.extend(new_slots_to_num)
    encoded_text_id = encoded_text['input_ids'].to(device)
    mask = encoded_text['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(encoded_text_id, mask)
    intent_probility = outputs[0].view(-1)
    _, intent_idx = torch.topk(intent_probility, k=2, dim=0)
    intent_idx = intent_idx.cpu()
    intent_num_probility = outputs[1].argmax()
    pred_intent_num.append(intent_num_probility.cpu())
    if intent_num_probility == 0:
        intent_idx = intent_idx[0]
    pred_intent = [1 if i in intent_idx else 0 for i in range(16)]
    pred_intent_label.append(list(pred_intent))
    slots_probility = outputs[2].argmax(dim=2).view(-1)
    pred_slots_num = slots_probility[1:-1].cpu()
    pred_token_slots.extend([t.item() for t in pred_slots_num])
    pred_token_slot = [find_key(slots_num, i)[0] for i in slots_probility]
    # print(f'new_tokens:{new_tokens}')
    # # print(f'pred_token_slot:{pred_token_slot}')
    pred_key_word = restore_keywords_from_tokens(char_level_tokens, pred_token_slot)
    # #print(f'pred_key_word:{pred_key_word}')
    pred_key_words.append(pred_key_word)

100%|██████████| 2943/2943 [03:27<00:00, 14.21it/s]


# Calculate metrics

In [60]:
def metric_compute(trues: list, preds: list):
    if len(trues) != len(preds):
        return 'Input lengthes not equal!'
    precision = 0
    precision_all = 0
    recall = 0
    recall_all = 0
    for true_label, pred_label in zip(trues, preds):
        # 查准率
        for pred in pred_label:
            if pred in true_label:
                precision += 1
            precision_all += 1
        # 查全率
        for true in true_label:
            if true in pred_label:
                recall += 1
            recall_all += 1
    P = precision/precision_all
    R = recall/recall_all
    F1 = 2 * P * R / (P + R)
    return P, R, F1

In [61]:
# print(true_intent_label)
# print(pred_intent_label)
#print(classification_report(true_intent_label, pred_intent_label,digits=6))
print(metric_compute(true_intent_label, pred_intent_label))

(1.0, 1.0, 1.0)


In [62]:
#pred_intent_num = pred_intent_num.cpu()

target_names = ['One Intent', 'Two Intent']
print(classification_report(true_intent_num , pred_intent_num, target_names=target_names))

              precision    recall  f1-score   support

  One Intent       1.00      1.00      1.00      2432
  Two Intent       1.00      1.00      1.00       511

    accuracy                           1.00      2943
   macro avg       1.00      1.00      1.00      2943
weighted avg       1.00      1.00      1.00      2943



In [63]:
print(true_key_words)
print(pred_key_words)

[[('南京市', 'city'), ('金浦假日百货(室内精品步行街)', 'POI'), ('3公里', 'distance'), ('在售', 'sale_status'), ('二手房', 'community_tag')], [('天津市', 'city'), ('河东实验教育集团·冠云小学', 'POI'), ('2公里', 'distance'), ('10%', 'green_rate'), ('在售', 'sale_status'), ('二手房', 'community_tag')], [('天津市', 'city'), ('宝坻区', 'district'), ('香江健康小镇玉兰园', 'community'), ('2公里', 'distance'), ('购物中心', 'POI:shopping_mall')], [('北京市', 'city'), ('北滨河路55号楼', 'community'), ('月坛南街北里', 'community'), ('市政宿舍', 'community'), ('北京市第四十四中学', 'POI')], [('天津市', 'city'), ('天津市南开中学滨海生态城学校', 'POI'), ('2公里', 'distance'), ('在售', 'sale_status'), ('新房', 'community_tag')], [('杭州市', 'city'), ('岸芷丁香庭', 'community'), ('映澜望宸轩', 'community'), ('翠铭轩', 'community'), ('浙窑公园', 'POI')], [('广州市', 'city'), ('花都区', 'district'), ('微澜苑', 'community'), ('北回归线山地运动主题公园', 'POI')], [('苏州市', 'city'), ('青云台', 'community'), ('科技城幸福里', 'community'), ('苏高新泉山雅境', 'community'), ('相城区中医医院门诊', 'POI')], [('南京市', 'city'), ('建邺区', 'district'), ('建发珺和府东区', 'community'), ('南京河西外国语学校', 'POI'),

In [64]:
city_count = 0
city_p = 0
district_count = 0
district_p = 0
POI_count = 0
POI_p = 0
community_count = 0
community_p = 0
shopping_mall_count = 0
shopping_mall_p = 0
supermarket_count = 0
supermarket_p = 0
hospital_count = 0
hospital_p = 0
park_count = 0
park_p = 0
subway_station_count = 0
subway_station_p = 0
bus_stop_count = 0
bus_stop_p = 0
primary_school_count = 0
primary_school_p = 0
middle_school_count = 0
middle_school_p = 0
POI_third_count = 0
POI_third_p = 0
sale_status_count = 0
sale_status_p = 0
community_tag_count = 0
community_tag_p = 0
distance_count = 0
distance_p = 0
house_price_count = 0
house_price_p = 0
green_rate_count = 0
green_rate_p = 0
acc_count = 0

for true_label, pred_label in zip(true_key_words, pred_key_words):
    true_city_value = []
    true_district_value = []
    true_POI_value = []
    true_community_value = []
    true_shopping_mall_value = []
    true_supermarket_value = []
    true_hospital_value = []
    true_park_value = []
    true_subway_station_value = []
    true_bus_stop_value = []
    true_primary_school_value = []
    true_middle_school_value = []
    true_POI_third_value = []
    true_sale_status_value = []
    true_community_tag_value = []
    true_distance_value = []
    true_house_price_value = []
    true_green_rate_value = []
    for key_word, slot in true_label:
        #print(slot)
        if slot == 'city':
            true_city_value.append(key_word)
            pred_city_value = [key[0] for key in pred_label if key[1] == 'city']
            
        elif slot == 'district':
            true_district_value.append(key_word)
            pred_district_value = [key[0] for key in pred_label if key[1] == 'district']
        
        elif slot == 'community':
            true_community_value.append(key_word)
            pred_community_value = [key[0] for key in pred_label if key[1] == 'community']    
        
        elif slot == 'POI':
            true_POI_value.append(key_word)
            pred_POI_value = [key[0] for key in pred_label if key[1] == 'POI']
            
        elif slot == 'shopping_mall':
            true_shopping_mall_value.append(key_word)
            pred_shopping_mall_value = [key[0] for key in pred_label if key[1] == 'POI:shopping_mall']
            
        elif slot == 'supermarket':
            true_supermarket_value.append(key_word)
            pred_supermarket_value = [key[0] for key in pred_label if key[1] == 'POI:supermarket']
            
        elif slot == 'hospital':
            true_hospital_value.append(key_word)
            pred_hospital_value = [key[0] for key in pred_label if key[1] == 'POI:hospital']
        
        elif slot == 'park':
            true_park_value.append(key_word)
            pred_park_value = [key[0] for key in pred_label if key[1] == 'POI:park']

        elif slot == 'subway_station':
            true_subway_station_value.append(key_word)
            pred_subway_station_value = [key[0] for key in pred_label if key[1] == 'POI:subway_station']
        
        elif slot == 'bus_stop':
            true_bus_stop_value.append(key_word)
            pred_bus_stop_value = [key[0] for key in pred_label if key[1] == 'POI:bus_stop']
        
        elif slot == 'primary_school':
            true_primary_school_value.append(key_word)
            pred_primary_school_value = [key[0] for key in pred_label if key[1] == 'POI:primary_school']

        elif slot == 'middle_school':
            true_middle_school_value.append(key_word)
            pred_middle_school_value = [key[0] for key in pred_label if key[1] == 'POI:middle_school']

        elif slot == 'POI_third':
            true_POI_third_value.append(key_word)
            pred_POI_third_value = [key[0] for key in pred_label if key[1] == 'POI_third']

        elif slot == 'sale_status':
            true_sale_status_value.append(key_word)
            pred_sale_status_value = [key[0] for key in pred_label if key[1] == 'sale_status']
        
        elif slot == 'community_tag':
            true_community_tag_value.append(key_word)
            pred_community_tag_value = [key[0] for key in pred_label if key[1] == 'community_tag']
        
        elif slot == 'distance':
            true_distance_value.append(key_word)
            pred_distance_value = [key[0] for key in pred_label if key[1] == 'distance']
        
        elif slot == 'house_price':
            true_house_price_value.append(key_word)
            pred_house_price_value = [key[0] for key in pred_label if key[1] == 'house_price']
        
        elif slot == 'green_rate':
            true_green_rate_value.append(key_word)
            pred_green_rate_value = [key[0] for key in pred_label if key[1] == 'green_rate']
    
    
    if true_city_value != []:
        if set(true_city_value) == set(pred_city_value):
            city_p += 1
        city_count += 1
    
    if true_district_value != []:
        if set(true_district_value) == set(pred_district_value):
            district_p += 1
        district_count += 1

    if true_POI_value != []:
        if set(true_POI_value) == set(pred_POI_value):
            POI_p += 1
        POI_count += 1
        
    if true_community_value != []:
        if set(true_community_value) == set(pred_community_value):
            community_p += 1
        community_count += 1

    if true_shopping_mall_value != []:
        if set(true_shopping_mall_value) == set(pred_shopping_mall_value):
            shopping_mall_p += 1
        shopping_mall_count += 1

    if true_supermarket_value != []:
        if set(true_supermarket_value) == set(pred_supermarket_value):
            supermarket_p += 1
        supermarket_count += 1

    if true_hospital_value != []:
        if set(true_hospital_value) == set(pred_hospital_value):
            hospital_p += 1
        hospital_count += 1
    
    if true_park_value != []:
        if set(true_park_value) == set(pred_park_value):
            park_p += 1
        park_count += 1
    
    if true_subway_station_value != []:
        if set(true_subway_station_value) == set(pred_subway_station_value):
            subway_station_p += 1
        subway_station_count += 1
    
    if true_bus_stop_value != []:
        if set(true_bus_stop_value) == set(pred_bus_stop_value):
            bus_stop_p += 1
        bus_stop_count += 1

    if true_primary_school_value != []:
        if set(true_primary_school_value) == set(pred_primary_school_value):
            primary_school_p += 1
        primary_school_count += 1
    
    if true_middle_school_value != []:
        if set(true_middle_school_value) == set(pred_middle_school_value):
            middle_school_p += 1
        middle_school_count += 1
    
    if true_POI_third_value != []:
        if set(true_POI_third_value) == set(pred_POI_third_value):
            POI_third_p += 1
        POI_third_count += 1
    
    if true_sale_status_value != []:
        if set(true_sale_status_value) == set(pred_sale_status_value):
            sale_status_p += 1
        sale_status_count += 1

    if true_community_tag_value != []:
        if set(true_community_tag_value) == set(pred_community_tag_value):
            community_tag_p += 1
        community_tag_count += 1
    
    if true_distance_value != []:
        if set(true_distance_value) == set(pred_distance_value):
            distance_p += 1
        distance_count += 1
    
    if true_house_price_value != []:
        if set(true_house_price_value) == set(pred_house_price_value):
            house_price_p += 1
        house_price_count += 1
    
    if true_green_rate_value != []:
        if set(true_green_rate_value) == set(pred_green_rate_value):
            green_rate_p += 1
        green_rate_count += 1
    

    if set(true_label) == set(pred_label):
        acc_count += 1

In [65]:
precision = {'city': 0, 'district': 0, 'POI': 0, 'community': 0, 
            'POI:shopping_mall': 0, 'POI:supermarket': 0, 'POI:hospital': 0,
            'POI:park': 0, 'POI:subway_station': 0, 'POI:bus_stop': 0,
            'POI:primary_school': 0, 'POI:middle_school': 0, 'POI_third': 0,
            'sale_status': 0, 'community_tag': 0, 'distance': 0,
            'house_price': 0, 'green_rate': 0, 'all': 0}

precision_all = {'city': 0, 'district': 0, 'POI': 0, 'community': 0, 
            'POI:shopping_mall': 0, 'POI:supermarket': 0, 'POI:hospital': 0,
            'POI:park': 0, 'POI:subway_station': 0, 'POI:bus_stop': 0,
            'POI:primary_school': 0, 'POI:middle_school': 0, 'POI_third': 0,
            'sale_status': 0, 'community_tag': 0, 'distance': 0,
            'house_price': 0, 'green_rate': 0, 'all': 0}

recall = {'city': 0, 'district': 0, 'POI': 0, 'community': 0, 
            'POI:shopping_mall': 0, 'POI:supermarket': 0, 'POI:hospital': 0,
            'POI:park': 0, 'POI:subway_station': 0, 'POI:bus_stop': 0,
            'POI:primary_school': 0, 'POI:middle_school': 0, 'POI_third': 0,
            'sale_status': 0, 'community_tag': 0, 'distance': 0,
            'house_price': 0, 'green_rate': 0, 'all': 0}

recall_all = {'city': 0, 'district': 0, 'POI': 0, 'community': 0, 
            'POI:shopping_mall': 0, 'POI:supermarket': 0, 'POI:hospital': 0,
            'POI:park': 0, 'POI:subway_station': 0, 'POI:bus_stop': 0,
            'POI:primary_school': 0, 'POI:middle_school': 0, 'POI_third': 0,
            'sale_status': 0, 'community_tag': 0, 'distance': 0,
            'house_price': 0, 'green_rate': 0, 'all': 0}

for true_label, pred_label in zip(true_key_words, pred_key_words):
    for pred in pred_label:
        slot = pred[1]
        keyword = pred[0]
        if pred in true_label:
            precision['all'] += 1
            precision[slot] += 1
        precision_all['all'] += 1
        precision_all[slot] += 1
    for true in true_label:
        slot = true[1]
        keyword = true[0]
        if true in pred_label:
            recall['all'] += 1
            recall[slot] += 1
        recall_all['all'] += 1
        recall_all[slot] += 1
Macro = {'P': 0, 'R': 0, 'F1': 0}
Micro = {'P': 0, 'R': 0, 'F1': 0}
for key in precision.keys():
    P = precision[key]/precision_all[key]
    if key != 'all':
        Macro['P'] += P
        Micro['P'] += P
    R = recall[key]/recall_all[key]
    if key != 'all':
        Macro['R'] += R
        Micro['R'] += R
    F1 = 2*P*R/(P+R)
Macro_P = Macro['P']/18
Macro_R = Macro['R']/18
Macro_F1 = 2*Macro_P*Macro_R/(Macro_P+Macro_R)
Micro_P = Micro['P']/18
Micro_R = Micro['R']/18
Micro_F1 = 2*Micro_P*Micro_R/(Micro_P+Micro_R)
print(f'Macro_P:{Macro_P}')
print(f'Macro_R:{Macro_R}')
print(f'Macro_F1:{Macro_F1}')
print(f'Micro_P:{Micro_P}')
print(f'Micro_R:{Micro_R}')
print(f'Micro_F1:{Micro_F1}')

Macro_P:0.9731550880444164
Macro_R:0.9576588735200681
Macro_F1:0.9653447964730947
Micro_P:0.9731550880444164
Micro_R:0.9576588735200681
Micro_F1:0.9653447964730947


### Save BERT's predicted SLU information

In [66]:
pred_key_words

[[('南京市', 'city'),
  ('金浦假日百货(室内精品步行街)', 'POI'),
  ('3公里', 'distance'),
  ('在售', 'sale_status'),
  ('二手房', 'community_tag')],
 [('天津市', 'city'),
  ('河东实验教育集团·冠云小学', 'POI'),
  ('2公里', 'distance'),
  ('10%', 'green_rate'),
  ('在售', 'sale_status'),
  ('二手房', 'community_tag')],
 [('天津市', 'city'),
  ('宝坻区', 'district'),
  ('香江健康小镇玉兰园', 'community'),
  ('2公里', 'distance'),
  ('购物中心', 'POI:shopping_mall')],
 [('北京市', 'city'),
  ('北滨河路55号楼', 'community'),
  ('月坛南街北里', 'community'),
  ('市政宿舍', 'community'),
  ('北京市第四十四中学', 'POI')],
 [('天津市', 'city'),
  ('天津市南开中学滨海生态城学校', 'POI'),
  ('2公里', 'distance'),
  ('在售', 'sale_status'),
  ('新房', 'community_tag')],
 [('杭州市', 'city'),
  ('岸芷丁香庭', 'community'),
  ('映澜望宸轩', 'community'),
  ('翠铭轩', 'community'),
  ('浙窑公园', 'POI')],
 [('广州市', 'city'),
  ('花都区', 'district'),
  ('微澜苑', 'community'),
  ('北回归线山地运动主题公园', 'POI')],
 [('苏州市', 'city'),
  ('青云台', 'community'),
  ('科技城幸福里', 'community'),
  ('苏高新泉山雅境', 'community'),
  ('相城区中医医院门诊', 'POI')],
 [('南京市', 'city

In [67]:
{'city': 0, 'district': 0, 'POI': 0, 'community': 0, 
            'POI:shopping_mall': 0, 'POI:supermarket': 0, 'POI:hospital': 0,
            'POI:park': 0, 'POI:subway_station': 0, 'POI:bus_stop': 0,
            'POI:primary_school': 0, 'POI:middle_school': 0, 'POI_third': 0,
            'sale_status': 0, 'community_tag': 0, 'distance': 0,
            'house_price': 0, 'green_rate': 0, 'all': 0}

def restore_keywords_from_tokens(tokens, token_slot):
    keywords = []
    current_tokens = []
    current_label = None
    token_slot = token_slot[1:-1]

    for token, slot in zip(tokens, token_slot):
        if slot.startswith('B-'):
            if current_tokens:
                keywords.append((''.join(current_tokens), current_label))
                current_tokens = []
            current_label = slot[2:]
            current_tokens.append(token)
        elif slot.startswith('I-') and current_label == slot[2:]:
            current_tokens.append(token)
        else:
            if current_tokens:
                keywords.append((''.join(current_tokens), current_label))
                current_tokens = []
                current_label = None

    if current_tokens:
        keywords.append((''.join(current_tokens), current_label))
    keyword_pair = []
    for keyword in keywords:
        if keyword[-1] == 'city':
            keyword_pair.append(f'城市:{keyword[0]}')
        elif keyword[-1] == 'district':
            keyword_pair.append(f'区域名称:{keyword[0]}')
        elif keyword[-1] == 'POI':
            keyword_pair.append(f'POI名称:{keyword[0]}')
        elif keyword[-1] == 'community':
            keyword_pair.append(f'小区名称:{keyword[0]}')
        elif keyword[-1] in ['POI:shopping_mall', 'POI:supermarket', 'POI:hospital',
                            'POI:park', 'POI:subway_station', 'POI:bus_stop',
                            'POI:primary_school', 'POI:middle_school']:
            keyword_pair.append(f'POI类型:{keyword[0]}')
        elif keyword[-1] == 'POI_third':
            keyword_pair.append(f'POI类型标签:{keyword[0]}')
        elif keyword[-1] == 'sale_status':
            keyword_pair.append(f'销售状态:{keyword[0]}')
        elif keyword[-1] == 'community_tag':
            keyword_pair.append(f'小区属性:{keyword[0]}')
        elif keyword[-1] == 'distance':
            keyword_pair.append(f'距离:{keyword[0]}')
        elif keyword[-1] == 'house_price':
            keyword_pair.append(f'成交均价:{keyword[0]}')
        elif keyword[-1] == 'green_rate':
            keyword_pair.append(f'绿化率:{keyword[0]}')

    return keyword_pair

In [68]:
json_file = './dataset/test.json'
with open(json_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)
all_intents = ['直线距离查询', '步行距离查询', '车行距离查询', '步行时间查询', '开车时间查询',
            '骑车时间查询', '公共交通时间查询', 'POI查询', '小区查询', '小区均价查询',
            '小区绿化率查询', '高峰期开车时间查询', '高峰期公共交通时间查询',
            '非高峰期开车时间查询', '非高峰期公共交通时间查询', '通勤时间查询']
for data in tqdm(datas):
    query = data['query']
    origin_slots = data['slots'].split(',')
    encoded_text = tokenizer(query, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(encoded_text['input_ids'][0])
    #new_tokens = align_tokens_with_query(tokens, query)
    new_tokens, _ = align_tokens_with_query(tokens, query, origin_slots)
    char_level_tokens = get_char_level_tokens(new_tokens)
    # print(f'new_tokens:{new_tokens}')
    encoded_text_id = encoded_text['input_ids'].to(device)
    mask = encoded_text['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(encoded_text_id, mask)
    intent_probility = outputs[0].view(-1)
    _, intent_idx = torch.topk(intent_probility, k=3, dim=0)
    intent_idx = intent_idx.cpu()
    intent_num = outputs[1].argmax().cpu()
    if intent_num == 0:
        intent_idx = intent_idx[0]
        intent = find_key(intents_num, intent_idx)[0]
    elif intent_num == 1:
        
        intent = [find_key(intents_num, i)[0] for i in intent_idx]
        intent = intent[0]+'+'+intent[1]
    elif intent_num == 2:
        intent = [find_key(intents_num, i)[0] for i in intent_idx]
        intent = intent[0]+'+'+intent[1] +'+'+intent[2]

    data["BERT_pred_intent"] = intent
    slots_probility = outputs[2].argmax(dim=2).view(-1)
    token_slot = [find_key(slots_num, i)[0] for i in slots_probility]
    pred_key_word = restore_keywords_from_tokens(char_level_tokens, token_slot)
    key_word = pred_key_word
    data["BERT_pred_slots"] = key_word

100%|██████████| 2943/2943 [03:26<00:00, 14.23it/s]


In [69]:
save_json_file = './results/test-with-BERT_pred_intent+slots.json'
with open(save_json_file, 'w', encoding='utf-8') as file:
    json.dump(datas, file, ensure_ascii=False, indent=4)